In [ ]:
import sys
import csv

from sklearn.svm import SVC
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB 
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

csv.field_size_limit(2**31-1)

In [ ]:
def get_data(t='test'):
    text= []
    label= []

    with open(f'../data-vermeer/{t}.csv') as fi:
        next(fi) # skips header row
        reader = csv.reader(fi, delimiter=',')

        for row in reader:
            text.append(row[0])
            label.append(row[1])

    return text, label

In [ ]:
X_test, y_test = get_data('test')
X_train, y_train = get_data('train')

In [ ]:
print(len(X_test), len(y_test))
print(len(X_train), len(y_train))   

In [ ]:
configurations = [('NB with Count', CountVectorizer(min_df=5, max_df=.5), MultinomialNB()),
                 ('NB with TfIdf', TfidfVectorizer(min_df=5, max_df=.5), MultinomialNB()),
                 ('LogReg with Count', CountVectorizer(min_df=5, max_df=.5), LogisticRegression(solver='liblinear')),
                 ('LogReg with TfIdf', TfidfVectorizer(min_df=5, max_df=.5), LogisticRegression(solver='liblinear')),
                 ('SVM with Count - rbf kernel', CountVectorizer(min_df=5, max_df=.5), SVC(kernel='rbf')),
                 ('SVM with Count - linear kernel', CountVectorizer(min_df=5, max_df=.5), SVC(kernel='linear')),
                 ('SVM with Tfidf - rbf kernel', TfidfVectorizer(min_df=5, max_df=.5), SVC(kernel='rbf')),
                 ('SVM with Tfidf - linear kernel', TfidfVectorizer(min_df=5, max_df=.5), SVC(kernel='linear')),
            # Added Random Forest classifiers:
                 ('Random Forest with Count', CountVectorizer(min_df=5, max_df=.5), RandomForestClassifier(n_estimators=100, random_state=42)),
                 ('Random Forest with TfIdf', TfidfVectorizer(min_df=5, max_df=.5), RandomForestClassifier(n_estimators=100, random_state=42)),
                 ]

for description, vectorizer, classifier in configurations:
    print(description)
    X_tr = vectorizer.fit_transform(X_train)
    X_te = vectorizer.transform(X_test)
    classifier.fit(X_tr, y_train)
    y_pred = classifier.predict(X_te)
    print(metrics.classification_report(y_test, y_pred) )
    print('\n')

#### now add a lemmatizer:
(Note I only keep the best of previous runs to save compute, is that smart?)

In [ ]:
# Import spaCy and load the English model
import spacy
nlp = spacy.load('en_core_web_sm')

# Define a spaCy lemmatizer tokenizer
def spacy_lemmatizer(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc]

configurations = [
    ('LogReg with Count', CountVectorizer(min_df=5, max_df=0.5), LogisticRegression(solver='saga')),
    ('SVM with Tfidf - linear kernel', TfidfVectorizer(min_df=5, max_df=0.5), SVC(kernel='linear')),
    # Add spaCy lemmatizer options
    ('LogReg with Count + spaCy lemmatizer', CountVectorizer(min_df=5, max_df=0.5, tokenizer=spacy_lemmatizer), LogisticRegression(solver='saga')),
    ('SVM with Tfidf - linear kernel + spaCy lemmatizer', TfidfVectorizer(min_df=5, max_df=0.5, tokenizer=spacy_lemmatizer), SVC(kernel='linear')),
]

for description, vectorizer, classifier in configurations:
    print(description)
    X_tr = vectorizer.fit_transform(X_train)
    X_te = vectorizer.transform(X_test)
    classifier.fit(X_tr, y_train)
    y_pred = classifier.predict(X_te)
    print(metrics.classification_report(y_test, y_pred))
    print('\n')

#### Note the new solver ('saga') selected for LogisticRegression had problems converging, which increased the time needed to run it, and results might be suboptimal (you could use the max_iter parameter (default=100) in LogisticRegression(max_iter=500,...) to allow more runs to try to converge)

#### also note its poorer performance with respect to the depricated solver ('liblinear') above

#### The complex spacy lemmatizer also adds substantively to the processing time -> looking at the results, is it worth it?

In [ ]:
#remember we didn't save the predictions of the best model, so let's do that now it was cleared after the end of the iteration...
#need to re-calculate the predictions of the best model

#compare the predicted labels of the best model to the true labels
description, vectorizer, classifier =  [('SVM with Tfidf - linear kernel', TfidfVectorizer(min_df=5, max_df=0.5), SVC(kernel='linear', probability=True))][0]

X_tr = vectorizer.fit_transform(X_train)
X_te = vectorizer.transform(X_test)
classifier.fit(X_tr, y_train)
y_pred = classifier.predict(X_te)

#now save this in a pd.DataFrame
import pandas as pd
results_df = pd.DataFrame({'text': X_test, 'true_label': y_test, 'predicted_label': y_pred})
#add a column whether the prediction was correct
results_df['correct'] = results_df['true_label'] == results_df['predicted_label']

#show the first few incorrect predictions per class:
for label in results_df['true_label'].unique():
    incorrect_predictions = results_df[(results_df['true_label'] == label) & (results_df['correct'] == False)]
    print(incorrect_predictions[['text', 'true_label', 'predicted_label']].head())

#get predicted probabilities for the best model
if hasattr(classifier, "predict_proba"):
    y_proba = classifier.predict_proba(X_te)
    #get the max probability for each prediction
    max_proba = y_proba.max(axis=1)
    results_df['max_proba'] = max_proba
    #show the first few incorrect predictions with the highest confidence
    incorrect_predictions = results_df[results_df['correct'] == False]
    high_confidence_incorrect = incorrect_predictions.sort_values(by='max_proba', ascending=False)
    print("High confidence incorrect predictions:\n", high_confidence_incorrect.loc[:, ['text', 'true_label', 'predicted_label', 'max_proba']].head())

In [ ]:
#make a pd.dataframe with the text, true label, predicted label, whether the prediction was correct, and the predicted probability for each label

#first get the predicted probabilities for each label
if hasattr(classifier, "predict_proba"):
    y_proba = classifier.predict_proba(X_te)
    #get the class labels
    class_labels = classifier.classes_
    #create a dataframe with the predicted probabilities for each label
    proba_df = pd.DataFrame(y_proba, columns=[f'proba_{label}' for label in class_labels])
    #concatenate this with the results_df
    results_df = pd.concat([results_df, proba_df], axis=1)  
    print(results_df.head())


#### OPTIONAL AND ADVANCED use the explainable AI package Shapley to help understand the model

### WARNING EXTREMELY SLOW EVEN ON A SMALL SAMPLE

In [ ]:
%pip install shap

In [ ]:
import shap

#pick the best model:
description, vectorizer, classifier =  [('SVM with Tfidf - linear kernel', TfidfVectorizer(min_df=5, max_df=0.5), SVC(kernel='linear', probability=True))][0]

X_tr = vectorizer.fit_transform(X_train)
X_te = vectorizer.transform(X_test)

# Fit your model as usual
classifier.fit(X_tr, y_train)

# Use the model's predict_proba for SHAP
explainer = shap.KernelExplainer(classifier.predict_proba, X_tr[:10])  # use a sample for background
shap_values = explainer.shap_values(X_te[:10])  # explain a few test samples

# Visualize
shap.summary_plot(shap_values, X_te[:10])